# IMDB Movie Review Sentiment Analyzer
## BIT4133: Natural Language Processing with Deep Learning

**Project:** Sentiment Analysis on IMDB Movie Reviews

**Weeks Covered:** 1, 2, and 3

This notebook covers:
- **Week 1:** Introduction to NLP, environment setup, tokenization, stopword removal
- **Week 2:** N-gram models and POS tagging for sentiment features
- **Week 3:** Hidden Markov Models, sequence labeling, and Named Entity Recognition
- **Week 4:** Syntactic and Semantic Analysis
- **Week 5:** CAT 1 Preparation and Mini NLP Project

---
# WEEK 1: Introduction to NLP and Applications
## Title: Introduction to NLP Concepts and Text Processing

### Fig 1: Python Installation Check
Verify Python is installed and running in Colab.

In [ ]:
# Fig 1: Python version check
import sys
print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Python executable: /usr/bin/python3


### Fig 2: Google Colab Setup
Confirm we are running inside Google Colab and check available resources.

In [ ]:
# Fig 2: Verify Colab environment
try:
    import google.colab
    print("Running on Google Colab")
except ImportError:
    print("Running on local Jupyter (Colab features unavailable)")

# Check available RAM and GPU
!cat /proc/meminfo | grep MemTotal
!nvidia-smi 2>/dev/null | head -15 || echo "No GPU enabled. Go to Runtime > Change runtime type > T4 GPU"

Running on Google Colab
MemTotal:       13286944 kB


### Fig 3: NLTK Installation
Install NLTK and download the resources we need for tokenization, stopwords, POS tagging, and NER.

In [ ]:
# Fig 3: Install NLTK and download required data
!pip install nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
nltk.download('treebank')
print("\nNLTK version:", nltk.__version__)
print("All NLTK resources downloaded successfully.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /r


NLTK version: 3.9.1
All NLTK resources downloaded successfully.


[nltk_data]   Unzipping corpora/treebank.zip.


### Load the IMDB Dataset
We use the built-in IMDB dataset from TensorFlow Keras. It contains 50,000 movie reviews labeled as positive (1) or negative (0).

In [ ]:
# Load IMDB dataset and reconstruct one sample review as plain text
from tensorflow.keras.datasets import imdb

# Load the dataset (keeps the top 10,000 most frequent words)
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)
print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print(f"Example label (1=positive, 0=negative): {y_train[0]}")

# Reconstruct readable text from the encoded review
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(encoded_review):
    # Offset by 3 because 0,1,2 are reserved for padding, start, and unknown
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

sample_review = decode_review(x_train[0])
print("\nSample review (first 500 chars):")
print(sample_review[:500])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Test samples: 25000
Example label (1=positive, 0=negative): 1
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Sample review (first 500 chars):
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to ever


### Fig 4: Tokenization Output
Break a review into individual word tokens using NLTK.

In [ ]:
# Fig 4: Tokenization
from nltk.tokenize import word_tokenize, sent_tokenize

# Use a clean sample review for clearer output
demo_review = ("This movie was absolutely fantastic! The acting was superb "
               "and the plot kept me engaged. However, the ending felt rushed "
               "and not as satisfying as I hoped.")

# Sentence tokenization
sentences = sent_tokenize(demo_review)
print("Sentence tokenization:")
for i, s in enumerate(sentences, 1):
    print(f"  {i}. {s}")

# Word tokenization
tokens = word_tokenize(demo_review)
print("\nWord tokenization:")
print(tokens)
print(f"\nTotal tokens: {len(tokens)}")

Sentence tokenization:
  1. This movie was absolutely fantastic!
  2. The acting was superb and the plot kept me engaged.
  3. However, the ending felt rushed and not as satisfying as I hoped.

Word tokenization:
['This', 'movie', 'was', 'absolutely', 'fantastic', '!', 'The', 'acting', 'was', 'superb', 'and', 'the', 'plot', 'kept', 'me', 'engaged', '.', 'However', ',', 'the', 'ending', 'felt', 'rushed', 'and', 'not', 'as', 'satisfying', 'as', 'I', 'hoped', '.']

Total tokens: 31


### Fig 5: Stopwords Removal Output
Remove common English stopwords so the model focuses on meaningful words.

In [ ]:
# Fig 5: Stopword removal
from nltk.corpus import stopwords
import string

stop_words = set(stopwords.words('english'))
print(f"Number of English stopwords: {len(stop_words)}")
print(f"Sample stopwords: {list(stop_words)[:15]}")

# Filter tokens: lowercase, remove stopwords, remove punctuation
filtered_tokens = [
    word.lower() for word in tokens
    if word.lower() not in stop_words and word not in string.punctuation
]

print("\nBEFORE stopword removal:")
print(tokens)
print(f"\nAFTER stopword removal:")
print(filtered_tokens)
print(f"\nTokens reduced from {len(tokens)} to {len(filtered_tokens)}")

Number of English stopwords: 198
Sample stopwords: ["she'd", "he's", 'o', 'as', 'haven', "he'd", 'am', 'before', 'shouldn', 'very', 'these', "that'll", 'isn', 'between', 'were']

BEFORE stopword removal:
['This', 'movie', 'was', 'absolutely', 'fantastic', '!', 'The', 'acting', 'was', 'superb', 'and', 'the', 'plot', 'kept', 'me', 'engaged', '.', 'However', ',', 'the', 'ending', 'felt', 'rushed', 'and', 'not', 'as', 'satisfying', 'as', 'I', 'hoped', '.']

AFTER stopword removal:
['movie', 'absolutely', 'fantastic', 'acting', 'superb', 'plot', 'kept', 'engaged', 'however', 'ending', 'felt', 'rushed', 'satisfying', 'hoped']

Tokens reduced from 31 to 14


### Fig 6: NLP Application Research
Document real-world NLP applications related to sentiment analysis.

In [ ]:
# Fig 6: Real-world NLP applications
applications = {
    "ChatGPT": "Conversational AI using transformer-based language models",
    "Google Translate": "Neural machine translation across 100+ languages",
    "Siri / Alexa": "Voice assistants using speech recognition and NLU",
    "Gmail Spam Filter": "Text classification to detect spam emails",
    "Amazon Reviews": "Sentiment analysis to summarize product feedback",
    "Rotten Tomatoes": "Aggregates movie review sentiment into scores",
    "Grammarly": "Grammar correction and writing style suggestions",
    "LinkedIn Search": "Named Entity Recognition for jobs, skills, companies"
}

print("Real-World NLP Applications:")
print("=" * 60)
for app, desc in applications.items():
    print(f"  • {app}: {desc}")
print("=" * 60)
print("\nThis project (IMDB sentiment analyzer) is most similar to")
print("Amazon Reviews and Rotten Tomatoes — text classification with")
print("sentiment labels.")

Real-World NLP Applications:
  • ChatGPT: Conversational AI using transformer-based language models
  • Google Translate: Neural machine translation across 100+ languages
  • Siri / Alexa: Voice assistants using speech recognition and NLU
  • Gmail Spam Filter: Text classification to detect spam emails
  • Amazon Reviews: Sentiment analysis to summarize product feedback
  • Rotten Tomatoes: Aggregates movie review sentiment into scores
  • Grammarly: Grammar correction and writing style suggestions
  • LinkedIn Search: Named Entity Recognition for jobs, skills, companies

This project (IMDB sentiment analyzer) is most similar to
Amazon Reviews and Rotten Tomatoes — text classification with
sentiment labels.


---
# WEEK 2: N-gram Models and POS Tagging
## Title: Language Modeling and Sentence Structure Analysis

### Fig 1: Bigram and Trigram Output
N-grams capture local word context. For sentiment, bigrams like "not good" carry meaning that single words miss.

In [ ]:
# Fig 1: Bigrams and trigrams
from nltk import bigrams, trigrams
from collections import Counter

review_text = ("The movie was not good at all. The story was predictable "
               "and the acting was not convincing. I would not recommend this film.")

tokens = word_tokenize(review_text.lower())

# Generate bigrams
bigram_list = list(bigrams(tokens))
print("Bigrams (first 10):")
for bg in bigram_list[:10]:
    print(f"  {bg}")

# Generate trigrams
trigram_list = list(trigrams(tokens))
print("\nTrigrams (first 5):")
for tg in trigram_list[:5]:
    print(f"  {tg}")

# Highlight sentiment-relevant negation bigrams
negation_bigrams = [bg for bg in bigram_list if bg[0] == 'not']
print("\nNegation bigrams (important for sentiment):")
for bg in negation_bigrams:
    print(f"  {bg}  <-- unigram model would miss this context")

Bigrams (first 10):
  ('the', 'movie')
  ('movie', 'was')
  ('was', 'not')
  ('not', 'good')
  ('good', 'at')
  ('at', 'all')
  ('all', '.')
  ('.', 'the')
  ('the', 'story')
  ('story', 'was')

Trigrams (first 5):
  ('the', 'movie', 'was')
  ('movie', 'was', 'not')
  ('was', 'not', 'good')
  ('not', 'good', 'at')
  ('good', 'at', 'all')

Negation bigrams (important for sentiment):
  ('not', 'good')  <-- unigram model would miss this context
  ('not', 'convincing')  <-- unigram model would miss this context
  ('not', 'recommend')  <-- unigram model would miss this context


### Fig 2: POS Tagging Example
Part-of-speech tags reveal each word's grammatical role. Adjectives (JJ) and adverbs (RB) carry the most sentiment.

In [ ]:
# Fig 2: POS tagging
from nltk import pos_tag

review = "The amazing actors delivered a brilliantly emotional performance."
tokens = word_tokenize(review)
tagged = pos_tag(tokens)

print("POS tagging result:")
print("-" * 50)
for word, tag in tagged:
    print(f"  {word:20s} -> {tag}")
print("-" * 50)
print("\nKey POS tags:")
print("  NN  = Noun, NNS = Plural Noun")
print("  JJ  = Adjective  (sentiment-carrying)")
print("  RB  = Adverb     (sentiment-carrying)")
print("  VB  = Verb,  VBD = Past tense verb")
print("  DT  = Determiner (the, a, an)")

POS tagging result:
--------------------------------------------------
  The                  -> DT
  amazing              -> JJ
  actors               -> NNS
  delivered            -> VBD
  a                    -> DT
  brilliantly          -> RB
  emotional            -> JJ
  performance          -> NN
  .                    -> .
--------------------------------------------------

Key POS tags:
  NN  = Noun, NNS = Plural Noun
  JJ  = Adjective  (sentiment-carrying)
  RB  = Adverb     (sentiment-carrying)
  VB  = Verb,  VBD = Past tense verb
  DT  = Determiner (the, a, an)


### Fig 3: Python Code for POS Tagging
Reusable function that POS-tags any review and reports the distribution of tags.

In [ ]:
# Fig 3: Reusable POS tagging function
def analyze_pos(text):
    """Tokenize text, POS-tag it, and return both the tagged tokens and the tag counts."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    tag_counts = Counter(tag for word, tag in tagged)
    return tagged, tag_counts

# Test on a positive and a negative example
positive = "This film was incredibly beautiful and deeply moving."
negative = "The plot was terrible and the dialogue was painfully boring."

for label, text in [("POSITIVE", positive), ("NEGATIVE", negative)]:
    print(f"\n--- {label} REVIEW ---")
    print(f"Text: {text}")
    tagged, counts = analyze_pos(text)
    print(f"Tagged: {tagged}")
    print(f"Tag distribution: {dict(counts)}")
    # Extract sentiment-bearing words (adjectives and adverbs)
    sentiment_words = [w for w, t in tagged if t.startswith('JJ') or t.startswith('RB')]
    print(f"Sentiment-bearing words (JJ/RB): {sentiment_words}")


--- POSITIVE REVIEW ---
Text: This film was incredibly beautiful and deeply moving.
Tagged: [('This', 'DT'), ('film', 'NN'), ('was', 'VBD'), ('incredibly', 'RB'), ('beautiful', 'JJ'), ('and', 'CC'), ('deeply', 'RB'), ('moving', 'VBG'), ('.', '.')]
Tag distribution: {'DT': 1, 'NN': 1, 'VBD': 1, 'RB': 2, 'JJ': 1, 'CC': 1, 'VBG': 1, '.': 1}
Sentiment-bearing words (JJ/RB): ['incredibly', 'beautiful', 'deeply']

--- NEGATIVE REVIEW ---
Text: The plot was terrible and the dialogue was painfully boring.
Tagged: [('The', 'DT'), ('plot', 'NN'), ('was', 'VBD'), ('terrible', 'JJ'), ('and', 'CC'), ('the', 'DT'), ('dialogue', 'NN'), ('was', 'VBD'), ('painfully', 'RB'), ('boring', 'JJ'), ('.', '.')]
Tag distribution: {'DT': 2, 'NN': 2, 'VBD': 2, 'JJ': 2, 'CC': 1, 'RB': 1, '.': 1}
Sentiment-bearing words (JJ/RB): ['terrible', 'painfully', 'boring']


### Fig 4: Language Prediction with N-grams
Use a bigram model to predict the next likely word given a context — a foundational language modeling task.

In [ ]:
# Fig 4: Bigram-based next-word prediction
from collections import defaultdict

# Build a small corpus from several reviews
corpus = (
    "the movie was great the story was great the acting was wonderful "
    "the movie was terrible the story was boring the acting was poor "
    "the film was amazing the plot was engaging the cast was talented"
)
tokens = word_tokenize(corpus.lower())

# Build bigram frequency model
bigram_model = defaultdict(Counter)
for w1, w2 in bigrams(tokens):
    bigram_model[w1][w2] += 1

def predict_next(word, top_n=3):
    """Return the top N most likely next words after the given word."""
    if word not in bigram_model:
        return []
    return bigram_model[word].most_common(top_n)

# Test predictions
print("Bigram language model — next word predictions:")
print("-" * 50)
for word in ['the', 'movie', 'was', 'story', 'acting']:
    predictions = predict_next(word)
    print(f"  After '{word}' -> {predictions}")

Bigram language model — next word predictions:
--------------------------------------------------
  After 'the' -> [('movie', 2), ('story', 2), ('acting', 2)]
  After 'movie' -> [('was', 2)]
  After 'was' -> [('great', 2), ('wonderful', 1), ('terrible', 1)]
  After 'story' -> [('was', 2)]
  After 'acting' -> [('was', 2)]


### Fig 5: Sentence Analysis Output
Full pipeline: sentence segmentation, tokenization, POS tagging, and sentiment-word extraction.

In [ ]:
# Fig 5: Combined sentence analysis
review = ("I really enjoyed this movie. The cinematography was breathtaking. "
          "However, the script was weak and predictable.")

print(f"Full review:\n  {review}\n")
print("=" * 60)

sentences = sent_tokenize(review)
for i, sentence in enumerate(sentences, 1):
    print(f"\nSentence {i}: {sentence}")
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)
    print(f"  Tokens: {tokens}")
    print(f"  POS tags: {tagged}")
    # Pull out adjectives and adverbs
    sentiment_words = [w for w, t in tagged if t.startswith('JJ') or t.startswith('RB')]
    print(f"  Sentiment words: {sentiment_words}")

Full review:
  I really enjoyed this movie. The cinematography was breathtaking. However, the script was weak and predictable.


Sentence 1: I really enjoyed this movie.
  Tokens: ['I', 'really', 'enjoyed', 'this', 'movie', '.']
  POS tags: [('I', 'PRP'), ('really', 'RB'), ('enjoyed', 'VBD'), ('this', 'DT'), ('movie', 'NN'), ('.', '.')]
  Sentiment words: ['really']

Sentence 2: The cinematography was breathtaking.
  Tokens: ['The', 'cinematography', 'was', 'breathtaking', '.']
  POS tags: [('The', 'DT'), ('cinematography', 'NN'), ('was', 'VBD'), ('breathtaking', 'VBG'), ('.', '.')]
  Sentiment words: []

Sentence 3: However, the script was weak and predictable.
  Tokens: ['However', ',', 'the', 'script', 'was', 'weak', 'and', 'predictable', '.']
  POS tags: [('However', 'RB'), (',', ','), ('the', 'DT'), ('script', 'NN'), ('was', 'VBD'), ('weak', 'JJ'), ('and', 'CC'), ('predictable', 'JJ'), ('.', '.')]
  Sentiment words: ['However', 'weak', 'predictable']


---
# WEEK 3: Hidden Markov Models and Sequence Labeling
## Title: Sequence Prediction and Hidden State Analysis

### Fig 1: Sequence Labeling Code
Sequence labeling assigns a tag to each token in a sequence. POS tagging is itself a sequence labeling task. Here we build a simple BIO-tagging scheme for sentiment phrases.

In [ ]:
# Fig 1: BIO sequence labeling for sentiment phrases
# B-POS = Beginning of positive phrase, I-POS = Inside, O = Outside

positive_words = {'great', 'wonderful', 'amazing', 'fantastic', 'brilliant', 'excellent'}
negative_words = {'terrible', 'awful', 'boring', 'poor', 'bad', 'worst'}

def bio_tag_sentiment(text):
    """Tag each token with a sentiment BIO label."""
    tokens = word_tokenize(text.lower())
    labels = []
    prev_label = 'O'
    for token in tokens:
        if token in positive_words:
            labels.append('B-POS' if prev_label != 'B-POS' else 'I-POS')
            prev_label = 'B-POS'
        elif token in negative_words:
            labels.append('B-NEG' if prev_label != 'B-NEG' else 'I-NEG')
            prev_label = 'B-NEG'
        else:
            labels.append('O')
            prev_label = 'O'
    return list(zip(tokens, labels))

review = "The acting was great but the script was terrible and boring."
result = bio_tag_sentiment(review)

print("BIO sequence labeling result:")
print("-" * 40)
for token, label in result:
    print(f"  {token:15s} -> {label}")

BIO sequence labeling result:
----------------------------------------
  the             -> O
  acting          -> O
  was             -> O
  great           -> B-POS
  but             -> O
  the             -> O
  script          -> O
  was             -> O
  terrible        -> B-NEG
  and             -> O
  boring          -> B-NEG
  .               -> O


### Fig 2: Hidden Markov Model Output
An HMM uses observed words to infer hidden states (such as POS tags). NLTK's HMM tagger learns transition and emission probabilities from a tagged corpus.

In [ ]:
# Fig 2: Train and use an HMM POS tagger
from nltk.tag import hmm
from nltk.corpus import treebank

# Use NLTK's Penn Treebank corpus for training
tagged_sentences = treebank.tagged_sents()
print(f"Training corpus size: {len(tagged_sentences)} sentences")
print(f"Sample tagged sentence:\n  {tagged_sentences[0][:10]}")

# Split into train and test
train_size = int(0.9 * len(tagged_sentences))
train_data = tagged_sentences[:train_size]
test_data = tagged_sentences[train_size:]

# Train the HMM tagger
trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train_supervised(train_data)
print("\nHMM tagger trained successfully.")

# Test on a movie review sentence
test_sentence = "The film was great and the actors were brilliant".split()
hmm_result = hmm_tagger.tag(test_sentence)

print("\nHMM tagging result on movie review:")
for word, tag in hmm_result:
    print(f"  {word:12s} -> {tag}")

Training corpus size: 3914 sentences
Sample tagged sentence:
  [('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT')]

HMM tagger trained successfully.


/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)



HMM tagging result on movie review:
  The          -> DT
  film         -> NN
  was          -> VBD
  great        -> JJ
  and          -> CC
  the          -> DT
  actors       -> NNP
  were         -> NNP
  brilliant    -> NNP


/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


### Fig 3: Named Entity Recognition Example
NER identifies people, organizations, and locations in text. For sentiment analysis, this lets us know *what* a reviewer is praising or criticizing — the actor, the director, or the studio.

In [ ]:
# Fig 3: Named Entity Recognition on a movie review
from nltk import ne_chunk

review = ("Christopher Nolan directed Inception starring Leonardo DiCaprio. "
          "Warner Bros released it in Los Angeles in July 2010.")

tokens = word_tokenize(review)
tagged = pos_tag(tokens)
named_entities = ne_chunk(tagged)

print("NER result (raw tree):")
print(named_entities)

# Pull out the named entities into a clean list
print("\nExtracted named entities:")
print("-" * 40)
for subtree in named_entities:
    if hasattr(subtree, 'label'):
        entity = " ".join(token for token, pos in subtree.leaves())
        print(f"  {entity:30s} -> {subtree.label()}")

NER result (raw tree):
(S
  (PERSON Christopher/NNP)
  (PERSON Nolan/NNP)
  directed/VBD
  Inception/NNP
  starring/VBG
  (PERSON Leonardo/NNP DiCaprio/NNP)
  ./.
  (ORGANIZATION Warner/NNP Bros/NNPS)
  released/VBD
  it/PRP
  in/IN
  (GPE Los/NNP Angeles/NNP)
  in/IN
  July/NNP
  2010/CD
  ./.)

Extracted named entities:
----------------------------------------
  Christopher                    -> PERSON
  Nolan                          -> PERSON
  Leonardo DiCaprio              -> PERSON
  Warner Bros                    -> ORGANIZATION
  Los Angeles                    -> GPE


### Fig 4: Sentence Labeling Results
Compare HMM-tagged sequences across positive and negative reviews. Notice the distribution of adjectives (JJ) and adverbs (RB).

In [ ]:
# Fig 4: Sentence labeling on labeled reviews
positive_reviews = [
    "The movie was absolutely fantastic and incredibly moving",
    "Brilliant performances and stunning cinematography throughout"
]

negative_reviews = [
    "The film was terribly boring and poorly written",
    "Disappointing acting and a confusing plot ruined this"
]

def label_sentence(sentence, label):
    print(f"\n[{label}] {sentence}")
    tokens = sentence.split()
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        marker = " <-- SENTIMENT" if tag.startswith(('JJ', 'RB')) else ""
        print(f"    {word:15s} -> {tag}{marker}")

print("=" * 60)
print("POSITIVE REVIEWS")
print("=" * 60)
for r in positive_reviews:
    label_sentence(r, "POS")

print("\n" + "=" * 60)
print("NEGATIVE REVIEWS")
print("=" * 60)
for r in negative_reviews:
    label_sentence(r, "NEG")

POSITIVE REVIEWS

[POS] The movie was absolutely fantastic and incredibly moving
    The             -> DT
    movie           -> NN
    was             -> VBD
    absolutely      -> RB <-- SENTIMENT
    fantastic       -> JJ <-- SENTIMENT
    and             -> CC
    incredibly      -> RB <-- SENTIMENT
    moving          -> VBG

[POS] Brilliant performances and stunning cinematography throughout
    Brilliant       -> JJ <-- SENTIMENT
    performances    -> NNS
    and             -> CC
    stunning        -> VBG
    cinematography  -> NN
    throughout      -> IN

NEGATIVE REVIEWS

[NEG] The film was terribly boring and poorly written
    The             -> DT
    film            -> NN
    was             -> VBD
    terribly        -> RB <-- SENTIMENT
    boring          -> JJ <-- SENTIMENT
    and             -> CC
    poorly          -> RB <-- SENTIMENT
    written         -> VBN

[NEG] Disappointing acting and a confusing plot ruined this
    Disappointing   -> VBG
    acting   

### Fig 5: Dataset Used for Training and Testing
Document the IMDB dataset structure: size, label distribution, and example samples.

In [ ]:
# Fig 5: IMDB dataset overview
import numpy as np

print("IMDB MOVIE REVIEW DATASET")
print("=" * 60)
print(f"Total reviews: {len(x_train) + len(x_test):,}")
print(f"  Training set: {len(x_train):,}")
print(f"  Test set:     {len(x_test):,}")

# Label distribution
train_pos = int(np.sum(y_train == 1))
train_neg = int(np.sum(y_train == 0))
print(f"\nLabel distribution (training):")
print(f"  Positive (1): {train_pos:,} ({train_pos/len(y_train)*100:.1f}%)")
print(f"  Negative (0): {train_neg:,} ({train_neg/len(y_train)*100:.1f}%)")

# Review length statistics
lengths = [len(r) for r in x_train]
print(f"\nReview length (words):")
print(f"  Min:    {min(lengths)}")
print(f"  Max:    {max(lengths)}")
print(f"  Mean:   {np.mean(lengths):.1f}")
print(f"  Median: {int(np.median(lengths))}")

# Example samples
print("\n" + "=" * 60)
print("SAMPLE POSITIVE REVIEW:")
pos_idx = np.where(y_train == 1)[0][0]
print(decode_review(x_train[pos_idx])[:300] + "...")

print("\nSAMPLE NEGATIVE REVIEW:")
neg_idx = np.where(y_train == 0)[0][0]
print(decode_review(x_train[neg_idx])[:300] + "...")

IMDB MOVIE REVIEW DATASET
Total reviews: 50,000
  Training set: 25,000
  Test set:     25,000

Label distribution (training):
  Positive (1): 12,500 (50.0%)
  Negative (0): 12,500 (50.0%)

Review length (words):
  Min:    11
  Max:    2494
  Mean:   238.7
  Median: 178

SAMPLE POSITIVE REVIEW:
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there wa...

SAMPLE NEGATIVE REVIEW:
? big hair big boobs bad music and a giant safety pin these are the words to best describe this terrible movie i love cheesy horror movies and i've seen hundreds but this had got to be on of the worst ever made the plot is paper thin and ridiculous the acting is an abomination the script is complete...


---
# WEEK 4: Syntactic and Semantic Analysis
## Title: Sentence Structure and Meaning Analysis using spaCy

### Setup: Install spaCy and download language models
spaCy is faster and more modern than NLTK for parsing. We need two models:
- `en_core_web_sm` — small model, fast, used for parsing
- `en_core_web_md` — medium model, has proper word vectors needed for similarity

In [ ]:
# Install spaCy (pre-installed on Colab, but ensures latest version)
!pip install -q spacy

# Download both language models
!python -m spacy download en_core_web_sm --quiet
!python -m spacy download en_core_web_md --quiet

import spacy
print(f"\nspaCy version: {spacy.__version__}")
print("Both language models downloaded successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 97.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 59.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

spaCy version: 3.8.14
Both language models downloaded successfully.


### Fig 1: spaCy Installation and Setup
Confirm both spaCy models load successfully.

In [ ]:
# Fig 1: Load both spaCy models
nlp_sm = spacy.load("en_core_web_sm")
nlp_md = spacy.load("en_core_web_md")

print("Small model (en_core_web_sm):")
print(f"  Pipeline: {nlp_sm.pipe_names}")
print(f"  Has word vectors: {nlp_sm.vocab.vectors_length > 0}")

print("\nMedium model (en_core_web_md):")
print(f"  Pipeline: {nlp_md.pipe_names}")
print(f"  Has word vectors: {nlp_md.vocab.vectors_length > 0}")
print(f"  Vector dimensions: {nlp_md.vocab.vectors_length}")

Small model (en_core_web_sm):
  Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
  Has word vectors: False

Medium model (en_core_web_md):
  Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
  Has word vectors: True
  Vector dimensions: 300


### Fig 2: Dependency Parsing — Practical Task 1
Analyze the grammatical structure of a sentence using spaCy. Dependency parsing reveals which word is the subject, verb, and object, and how words relate.

In [ ]:
# Fig 2: Dependency parsing (Practical Task 1)
text = "The lecturer teaches Natural Language Processing."
doc = nlp_sm(text)

print(f"Input sentence: {text}\n")
print(f"{'TOKEN':<20} {'DEPENDENCY':<15} {'HEAD':<15} {'POS':<10}")
print("-" * 60)
for token in doc:
    print(f"{token.text:<20} {token.dep_:<15} {token.head.text:<15} {token.pos_:<10}")

# Identify subject, verb, object
print("\nKey grammatical roles:")
for token in doc:
    if token.dep_ == "nsubj":
        print(f"  Subject:      {token.text}")
    elif token.dep_ == "ROOT":
        print(f"  Root verb:    {token.text}")
    elif token.dep_ in ("dobj", "pobj"):
        print(f"  Object:       {token.text}")

Input sentence: The lecturer teaches Natural Language Processing.

TOKEN                DEPENDENCY      HEAD            POS       
------------------------------------------------------------
The                  det             lecturer        DET       
lecturer             nsubj           teaches         NOUN      
teaches              ROOT            teaches         VERB      
Natural              compound        Language        PROPN     
Language             compound        Processing      PROPN     
Processing           dobj            teaches         PROPN     
.                    punct           teaches         PUNCT     

Key grammatical roles:
  Subject:      lecturer
  Root verb:    teaches
  Object:       Processing


### Fig 3: Dependency Parsing on a Movie Review
Apply the same technique to an IMDB-style review to connect this week's work back to the sentiment analysis project.

In [ ]:
# Fig 3: Dependency parsing on a movie review
review = "The brilliant director crafted an emotional masterpiece."
doc = nlp_sm(review)

print(f"Review: {review}\n")
print(f"{'TOKEN':<15} {'DEP':<12} {'HEAD':<15} {'POS':<8} {'EXPLANATION'}")
print("-" * 80)
for token in doc:
    explanation = spacy.explain(token.dep_) or ""
    print(f"{token.text:<15} {token.dep_:<12} {token.head.text:<15} {token.pos_:<8} {explanation}")

# Visualize the parse tree as text
print("\nDependency tree structure:")
def show_tree(token, depth=0):
    print("  " * depth + f"+- {token.text} ({token.dep_})")
    for child in token.children:
        show_tree(child, depth + 1)

root = [token for token in doc if token.dep_ == "ROOT"][0]
show_tree(root)

Review: The brilliant director crafted an emotional masterpiece.

TOKEN           DEP          HEAD            POS      EXPLANATION
--------------------------------------------------------------------------------
The             det          director        DET      determiner
brilliant       amod         director        ADJ      adjectival modifier
director        nsubj        crafted         NOUN     nominal subject
crafted         ROOT         crafted         VERB     root
an              det          masterpiece     DET      determiner
emotional       amod         masterpiece     ADJ      adjectival modifier
masterpiece     dobj         crafted         NOUN     direct object
.               punct        crafted         PUNCT    punctuation

Dependency tree structure:
+- crafted (ROOT)
  +- director (nsubj)
    +- The (det)
    +- brilliant (amod)
  +- masterpiece (dobj)
    +- an (det)
    +- emotional (amod)
  +- . (punct)


### Fig 4: Semantic Similarity — Practical Task 2
Compare meaning similarity between sentences using spaCy's word vectors. Note: we use `en_core_web_md` here because it has proper word vectors trained on a large corpus.

In [ ]:
# Fig 4: Semantic similarity (Practical Task 2)
sentence1 = nlp_md("The student passed the examination")
sentence2 = nlp_md("The learner succeeded in the exam")

similarity = sentence1.similarity(sentence2)

print(f"Sentence 1: '{sentence1.text}'")
print(f"Sentence 2: '{sentence2.text}'")
print(f"\nSimilarity Score: {similarity:.4f}")

# Interpret the score
if similarity > 0.85:
    interpretation = "Very similar (near-paraphrase)"
elif similarity > 0.65:
    interpretation = "Moderately similar (related meaning)"
elif similarity > 0.40:
    interpretation = "Weakly similar (loose topical link)"
else:
    interpretation = "Unrelated (different topics)"
print(f"Interpretation:   {interpretation}")

Sentence 1: 'The student passed the examination'
Sentence 2: 'The learner succeeded in the exam'

Similarity Score: 0.9391
Interpretation:   Very similar (near-paraphrase)


### Fig 5: Class Assignment — Dependency Analysis and Semantic Comparison
Run the two class assignment tasks: dependency parsing for two given sentences, and semantic similarity for similar vs unrelated sentence pairs.

In [ ]:
# Fig 5a: Assignment 1 — Dependency analysis for two sentences
assignment_sentences = [
    "The administrator updated student records",
    "Machine learning improves language processing"
]

for i, sent in enumerate(assignment_sentences, 1):
    print(f"\n=== Sentence {i}: {sent} ===")
    doc = nlp_sm(sent)
    print(f"{'TOKEN':<18} {'DEP':<12} {'HEAD':<18} {'POS':<8}")
    print("-" * 60)
    for token in doc:
        print(f"{token.text:<18} {token.dep_:<12} {token.head.text:<18} {token.pos_:<8}")
    # Summary
    subj = [t.text for t in doc if t.dep_ == "nsubj"]
    verb = [t.text for t in doc if t.dep_ == "ROOT"]
    obj = [t.text for t in doc if t.dep_ in ("dobj", "pobj")]
    print(f"Subject: {subj} | Verb: {verb} | Object: {obj}")


=== Sentence 1: The administrator updated student records ===
TOKEN              DEP          HEAD               POS     
------------------------------------------------------------
The                det          administrator      DET     
administrator      nsubj        updated            NOUN    
updated            ROOT         updated            VERB    
student            compound     records            NOUN    
records            dobj         updated            NOUN    
Subject: ['administrator'] | Verb: ['updated'] | Object: ['records']

=== Sentence 2: Machine learning improves language processing ===
TOKEN              DEP          HEAD               POS     
------------------------------------------------------------
Machine            compound     learning           NOUN    
learning           nsubj        improves           NOUN    
improves           ROOT         improves           VERB    
language           compound     processing         NOUN    
processing         

In [ ]:
# Fig 5b: Assignment 2 — Semantic similarity for similar vs unrelated pairs
print("=" * 65)
print("PAIR 1 — Two SIMILAR sentences")
print("=" * 65)
s1 = nlp_md("The movie was fantastic and very entertaining")
s2 = nlp_md("The film was excellent and highly enjoyable")
sim1 = s1.similarity(s2)
print(f"Sentence A: '{s1.text}'")
print(f"Sentence B: '{s2.text}'")
print(f"Similarity score: {sim1:.4f}")

print("\n" + "=" * 65)
print("PAIR 2 — Two UNRELATED sentences")
print("=" * 65)
s3 = nlp_md("The movie was fantastic and very entertaining")
s4 = nlp_md("The car needs an oil change next week")
sim2 = s3.similarity(s4)
print(f"Sentence A: '{s3.text}'")
print(f"Sentence B: '{s4.text}'")
print(f"Similarity score: {sim2:.4f}")

print("\n" + "=" * 65)
print("COMPARISON")
print("=" * 65)
print(f"Similar pair score:   {sim1:.4f}")
print(f"Unrelated pair score: {sim2:.4f}")
print(f"Difference:           {sim1 - sim2:.4f}")
print("\nThe similar pair scores higher because the sentences share")
print("semantic context (movie review domain, positive sentiment words).")

PAIR 1 — Two SIMILAR sentences
Sentence A: 'The movie was fantastic and very entertaining'
Sentence B: 'The film was excellent and highly enjoyable'
Similarity score: 0.9192

PAIR 2 — Two UNRELATED sentences
Sentence A: 'The movie was fantastic and very entertaining'
Sentence B: 'The car needs an oil change next week'
Similarity score: 0.7706

COMPARISON
Similar pair score:   0.9192
Unrelated pair score: 0.7706
Difference:           0.1486

The similar pair scores higher because the sentences share
semantic context (movie review domain, positive sentiment words).


---
# WEEK 5: CAT 1 Preparation and Mini NLP Project
## Title: Complete NLP Pipeline and Student Academic Assistant Chatbot

### Fig 1: Complete NLP Pipeline — Practical Task 1
Integrate everything learned in Weeks 1-4 into a single text-processing pipeline: tokenization, stopword removal, POS tagging, and (added) dependency parsing.

In [ ]:
# Fig 1: Complete NLP pipeline
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag

text = "Natural Language Processing helps computers understand language."

print(f"INPUT TEXT:\n  {text}\n")

# Step 1: Tokenization
words = word_tokenize(text)
print(f"STEP 1 — Tokens:\n  {words}\n")

# Step 2: Stopword removal
stop_words = set(stopwords.words('english'))
filtered = [w for w in words if w.lower() not in stop_words]
print(f"STEP 2 — After stopword removal:\n  {filtered}\n")

# Step 3: POS tagging
pos = pos_tag(filtered)
print(f"STEP 3 — POS tagged:\n  {pos}\n")

# Step 4: Dependency parsing (using spaCy on the original text)
doc = nlp_sm(text)
print("STEP 4 — Dependency relations:")
for token in doc:
    if token.dep_ != "punct":
        print(f"  {token.text:<15} --{token.dep_}--> {token.head.text}")

INPUT TEXT:
  Natural Language Processing helps computers understand language.

STEP 1 — Tokens:
  ['Natural', 'Language', 'Processing', 'helps', 'computers', 'understand', 'language', '.']

STEP 2 — After stopword removal:
  ['Natural', 'Language', 'Processing', 'helps', 'computers', 'understand', 'language', '.']

STEP 3 — POS tagged:
  [('Natural', 'JJ'), ('Language', 'NNP'), ('Processing', 'NNP'), ('helps', 'VBZ'), ('computers', 'NNS'), ('understand', 'JJ'), ('language', 'NN'), ('.', '.')]

STEP 4 — Dependency relations:
  Natural         --compound--> Language
  Language        --compound--> Processing
  Processing      --nsubj--> helps
  helps           --ROOT--> helps
  computers       --nsubj--> understand
  understand      --ccomp--> helps
  language        --dobj--> understand


### Fig 2: Rule-Based Chatbot Simulation — Practical Task 2
Build a simple keyword-matching chatbot. We use a function-based design so the chatbot can be tested with predefined inputs (for screenshots) and then run interactively.

In [ ]:
# Fig 2: Rule-based chatbot core logic
def simple_chatbot_response(user_input):
    """Return the bot's response to a user message based on keyword matching."""
    text = user_input.lower()

    if "hello" in text or "hi" in text:
        return "Hello Student"
    elif "exam" in text:
        return "CAT 1 starts next week"
    elif "bye" in text:
        return "Goodbye"
    else:
        return "I do not understand"

# Simulate a conversation with predefined inputs (for the screenshot)
print("Welcome to Student Help Chatbot\n")
test_messages = [
    "Hello",
    "When is the exam?",
    "Can you sing for me?",
    "Bye"
]
for msg in test_messages:
    print(f"You: {msg}")
    response = simple_chatbot_response(msg)
    print(f"Bot: {response}\n")
    if "bye" in msg.lower():
        break

Welcome to Student Help Chatbot

You: Hello
Bot: Hello Student

You: When is the exam?
Bot: CAT 1 starts next week

You: Can you sing for me?
Bot: I do not understand

You: Bye
Bot: Goodbye



### Mini Project: Student Academic Assistant Chatbot
Extended version that handles greetings, CAT schedule questions, unit registration questions, and a clean exit. Uses NLP tokenization to match intents more robustly than simple substring checks.

### Fig 3: Student Academic Assistant Chatbot — Code
Intent-based chatbot with an intent dictionary, tokenized matching, and unknown-input fallback.

In [ ]:
# Fig 3: Student Academic Assistant Chatbot

# Intent definitions — each intent has trigger keywords and a response
intents = {
    "greeting": {
        "keywords": ["hello", "hi", "hey", "greetings", "morning", "afternoon"],
        "response": "Hello! I am your Student Academic Assistant. How can I help you today?"
    },
    "cat_schedule": {
        "keywords": ["cat", "exam", "test", "assessment", "schedule", "when"],
        "response": "CAT 1 is scheduled for next week. Please check the student portal for the exact date and venue."
    },
    "unit_registration": {
        "keywords": ["register", "registration", "unit", "course", "enroll", "sign up"],
        "response": "To register for units, log in to the student portal, go to 'My Units', select your semester units, and click submit before the deadline."
    },
    "fees": {
        "keywords": ["fee", "fees", "payment", "pay", "tuition"],
        "response": "For fee balances and payment, visit the finance office or check your portal under 'Finance'."
    },
    "results": {
        "keywords": ["result", "results", "grade", "grades", "marks", "transcript"],
        "response": "Results are released on the student portal at the end of each semester."
    },
    "goodbye": {
        "keywords": ["bye", "goodbye", "exit", "quit", "stop", "end"],
        "response": "Goodbye! Good luck with your studies."
    }
}

def detect_intent(user_input):
    """Tokenize the input, lowercase it, and match against intent keywords."""
    tokens = set(word_tokenize(user_input.lower()))
    for intent_name, intent_data in intents.items():
        if any(keyword in tokens or keyword in user_input.lower() for keyword in intent_data["keywords"]):
            return intent_name
    return None

def chatbot_respond(user_input):
    """Generate a response based on detected intent."""
    intent = detect_intent(user_input)
    if intent is None:
        return None, "I'm not sure I understand. You can ask me about CAT schedule, unit registration, fees, or results."
    return intent, intents[intent]["response"]

print("Student Academic Assistant Chatbot is ready.")
print(f"Supported intents: {list(intents.keys())}")

Student Academic Assistant Chatbot is ready.
Supported intents: ['greeting', 'cat_schedule', 'unit_registration', 'fees', 'results', 'goodbye']


### Fig 4: Mini Project — Chatbot Interaction Demo
Run a simulated conversation showing the chatbot handling all required scenarios: greeting, CAT question, unit registration question, an unknown input, and exit.

In [ ]:
# Fig 4: Simulated conversation covering all required scenarios
print("=" * 65)
print("STUDENT ACADEMIC ASSISTANT CHATBOT — CONVERSATION DEMO")
print("=" * 65)

demo_conversation = [
    "Hello there",
    "When is the CAT 1 exam?",
    "How do I register for my units?",
    "Can I check my fee balance?",
    "What about results?",
    "Tell me a joke",
    "Goodbye"
]

for user_msg in demo_conversation:
    print(f"\nYou: {user_msg}")
    intent, response = chatbot_respond(user_msg)
    intent_tag = f"[intent: {intent}]" if intent else "[intent: unknown]"
    print(f"Bot: {response}")
    print(f"     {intent_tag}")
    if intent == "goodbye":
        print("\n[Chatbot session ended]")
        break

STUDENT ACADEMIC ASSISTANT CHATBOT — CONVERSATION DEMO

You: Hello there
Bot: Hello! I am your Student Academic Assistant. How can I help you today?
     [intent: greeting]

You: When is the CAT 1 exam?
Bot: CAT 1 is scheduled for next week. Please check the student portal for the exact date and venue.
     [intent: cat_schedule]

You: How do I register for my units?
Bot: To register for units, log in to the student portal, go to 'My Units', select your semester units, and click submit before the deadline.
     [intent: unit_registration]

You: Can I check my fee balance?
Bot: For fee balances and payment, visit the finance office or check your portal under 'Finance'.
     [intent: fees]

You: What about results?
Bot: Results are released on the student portal at the end of each semester.
     [intent: results]

You: Tell me a joke
Bot: I'm not sure I understand. You can ask me about CAT schedule, unit registration, fees, or results.
     [intent: unknown]

You: Goodbye
Bot: Goodbye! G

### Fig 5: Interactive Chatbot (Optional — Run if You Want to Try It Yourself)
The cell below runs the chatbot in interactive mode. It will keep asking for input until you type 'bye'.

In [ ]:
# Fig 5: Interactive version — uncomment the lines below to run
# Note: Colab's input() works but blocks the notebook. Type 'bye' to exit.

# print("Welcome to the Student Academic Assistant Chatbot")
# print("Type 'bye' to exit.\n")
# while True:
#     user = input("You: ")
#     if not user.strip():
#         continue
#     intent, response = chatbot_respond(user)
#     print(f"Bot: {response}")
#     if intent == "goodbye":
#         break

print("To use the interactive chatbot, uncomment the code above and run this cell.")

To use the interactive chatbot, uncomment the code above and run this cell.


---
## End of Weeks 1–5

**Completed:**
- Week 1: Environment setup, tokenization, stopwords
- Week 2: N-grams, POS tagging, language modeling
- Week 3: HMMs, sequence labeling, NER
- Week 4: Dependency parsing and semantic similarity (spaCy)
- Week 5: Full NLP pipeline + Student Academic Assistant Chatbot mini-project

**Planned Future Work (Weeks 6+):**
- Word embeddings (Word2Vec, GloVe)
- LSTM model for sentiment classification on IMDB
- Fine-tuning DistilBERT
- Gradio demo deployment
